In [1]:
import folium
korea_map =folium.Map(location=[37.5666, 126.9784], tiles="Cartodb Positron", zoom_start=12)


In [2]:
korea_map

In [3]:
import requests
payload = {"in_biz_cds":"0",
"in_scodes":"0",
"ins_lat":"37.56682",
"ins_lng":"126.97865",
"search_text":"",
"p_sido_cd":"01",
"p_gugun_cd":"",
"in_distance":"0",
"in_biz_cd":"",
"isError":"true",
"searchType":"C",
"set_date":"",
"all_store":"0",
"T03":"0",
"T01":"0",
"T27":"0",
"T12":"0",
"T09":"0",
"T30":"0",
"T05":"0",
"T22":"0",
"T21":"0",
"T36":"0",
"T43":"0",
"Z9999":"0",
"T64":"0",
"P02":"0",
"P10":"0",
"P50":"0",
"P20":"0",
"P60":"0",
"P30":"0",
"P70":"0",
"P40":"0",
"P80":"0",
"whcroad_yn":"0",
"P90":"0",
"P01":"0",
"new_bool":"0",
"iend":"1000",
"rndCod":"ZI20ETDAT8",}


In [4]:
starbucks = "https://www.starbucks.co.kr/store/getStore.do?r=4RVFP1RE84"

In [5]:
starbucks_address = requests.post(starbucks, data=payload).text

In [6]:
import json
star_dict1 = json.loads(starbucks_address)


In [11]:
import json
star_dict1 = json.loads(starbucks_address)

In [13]:
for x in star_dict1['list']:
    folium.Marker(location=[x['lat'], x['lot']], popup=x['s_name']).add_to(korea_map)


In [15]:
korea_map

In [17]:
import pandas as pd
pd.DataFrame(star_dict1['list'])[['gugun_name']].value_counts(ascending=False)

gugun_name
강남구           104
서초구            60
중구             54
영등포구           47
송파구            42
종로구            42
마포구            38
강서구            35
광진구            24
서대문구           24
용산구            24
강동구            21
양천구            19
성북구            18
은평구            17
성동구            17
관악구            16
금천구            15
노원구            15
동작구            14
구로구            13
동대문구           13
중랑구            10
도봉구             8
강북구             7
Name: count, dtype: int64

In [18]:
import json
geo_data = json.load(open("./data/skorea_municipalities_geo_simple.json", "r"))


In [21]:
star_agg = pd.DataFrame(star_dict1['list'])[['gugun_name']].value_counts(ascending=False)


seoul_map = folium.Map(location=[37.55, 126.88],  tiles="Cartodb Positron", zoom_start=12)


In [22]:
g_map = folium.Choropleth(
    geo_data=geo_data, data = star_agg, 
    fill_color="YlOrRd", fill_opacity=0.7, line_opacity=0.3, key_on='feature.properties.name'
)


In [26]:
star_df = pd.DataFrame(star_dict1['list'])
star_agg = star_df.groupby('gugun_name')[['seq']].count().rename(columns={'seq' : "count"})
seoul_map = folium.Map(location=[37.55, 126.88],  tiles="Cartodb Positron", zoom_start=12)
g_map = folium.Choropleth(geo_data=geo_data, data=star_agg['count'],
                  fill_color='YlOrRdimport pickle
with open("./data/news.pkl", "rb") as f:
    data = pickle.load(f)
', fill_opacity=0.7, line_opacity=0.3, key_on = 'feature.properties.name'
                  )
g_map.add_to(seoul_map)


In [27]:
seoul_map

In [30]:
import pickle
with open("./data/news.pkl", "rb") as f:
    data = pickle.load(f)


In [32]:
from langchain_ollama import OllamaEmbeddings

embedding = OllamaEmbeddings(
    model="embeddinggemma:300m",
    base_url="http://host.docker.internal:11434"
)
# embedding.embed_query("대한민국")


In [34]:
from langchain_postgres import PGEngine, PGVectorStore
import os 


In [35]:
DB_USER = os.getenv("DB_USER", "langchain")
DB_PASSWORD = os.getenv("DB_PASSWORD", "langchain")
DB_HOST = os.getenv("DB_HOST", "postgres")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "langchain")

CONNECTION_STRING = (
    f"postgresql+psycopg://"
    f"{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = PGEngine.from_connection_string(
    url=CONNECTION_STRING,
)


In [36]:
vector_store = PGVectorStore.create_sync(
    engine=engine,
    table_name="news",
    embedding_service=embedding
)

In [37]:
from langchain_core.load import dumps, loads

In [40]:
data2_dumps = list(set([dumps(x) for x in data]))

In [43]:
data3 = [loads(x) for x in data2_dumps]

/tmp/ipykernel_341/3134092216.py:1: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  data3 = [loads(x) for x in data2_dumps]


In [45]:
vector_store.add_documents(
    documents=data3
)

['21a6697d-9286-44f0-9a5f-d23b16a2b864',
 '91351476-0a13-454e-8d01-f24ad16f364f',
 '98714581-1432-4305-a162-f88f53430d7c',
 '34b1bf33-6b37-4350-9794-091467d40230',
 '3e0e42d5-9ce7-423b-8862-407038380a98',
 '5482623c-cae9-44d9-ba63-b1053f249a54',
 '4b9af62f-8f30-43ed-86c2-2417289b517a',
 '6f7ef953-a2e8-4f64-a5ba-98878675d3ca',
 'e0b8e406-7c3b-417b-b85f-3fa7074b4fdb',
 '75df36ab-cef8-49ab-ab62-33c09588ef9e',
 '4755c90d-df37-43cd-baa5-71865287f595',
 'e0aa841c-fb12-4c6a-825a-a0ae568e793d',
 'ab3e6d97-878f-450b-8d74-3eb0a1c6f620',
 'b51deb06-c2a1-451f-9979-ecd0a69aa84c',
 '0a3f0406-c342-40a3-b419-43a9b86fcb7d',
 'afb43a63-68ef-4125-a7df-c0d558292b8d',
 '0ef4287b-b581-4dfe-a763-046aacc1b9f3',
 '5c5a0b00-2a0e-47dd-bfc1-6fb3937f20a7',
 '017c62d6-8c61-49c3-ac90-566be8180815',
 'c47e2c29-f09e-4a9c-a035-37e64753d7e7',
 'ae3a2f82-b48c-4198-bbad-914db15fb151',
 'ab9c2b82-fcd8-4676-9d2d-5b5632185d10',
 'fe347a5d-f2d6-49a4-967d-1d9e10e3836a',
 '9e517707-1c96-41f6-9bd8-27c7c3906646',
 '4e191e39-ad28-

In [46]:
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k' : 3})

In [47]:
retriever.invoke("비트코인 전망")

[Document(id='5c5a0b00-2a0e-47dd-bfc1-6fb3937f20a7', metadata={'source': 'https://n.news.naver.com/mnews/article/421/0009069915'}, page_content='\n\n\n\n\n(유니티 제공)/뉴스1(서울=뉴스1) 김정현 기자 = 게임 엔진 유니티가 차세대 제작 플랫폼 \'유니티7\'(Unity 7)의 개발 로드맵을 공개했다.유니티는 21일 오전 서울 코엑스에서 글로벌 개발자 컨퍼런스 \'유나이트 서울 2026\'을 열고 차세대 제작 플랫폼 유니티7 관련 계획을 발표했다.유니티에 따르면 유니티7은 △더욱 빠른 제작 △개방형 협업 생태계 △획기적인 그래픽 △더욱 스마트해진 성장 및 수익화 △호환성 유지 5가지 핵심 축을 기반으로 구축됐다.유니티7은 개방형 협업 플랫폼을 통해 개발자, 아티스트, 프로듀서, 코딩 에이전트가 전체 개발 라이프사이클에 걸쳐 함께 작업할 수 있도록 지원하는 것을 특징으로 한다.개발자가 이미 사용하고 있는 AI 도구와도 긴밀하게 연동될 예정이다. 또 소규모 개발팀에게도 게임 수익화를 원활히 할 수 있도록 하는 기능도 탑재했다.마지막으로 유니티7은 기존 유니티6의 구조를 그대로 계승해 엔진 전환 과정에서 기존 작업물에 문제가 생기지 않도록 했다.이번 유니티7은 오는 12월 초기 베타 테스트에 들어간다. 정식 버전은 오는 2027년 1분기에 출시될 예정이다.매튜 브롬버그(Matthew Bromberg) 유니티 최고경영자(CEO)는 "이제 게임 개발의 미래는 가장 규모가 큰 개발팀이 아니라 새로운 기술로 고유한 콘텐츠를 만들고 플레이어를 확보할 수 있는 팀의 것이 될 것"이라며 "유니티7은 이같은 시대적 요구에 부응하는 플랫폼"이라고 강조했다.\n\t\t'),
 Document(id='8319a84b-66f3-4176-a685-5fd084a497e0', metadata={'source': 'https://n.news.naver.com/mnews/

In [48]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain  # 검색된 문서를 프롬프트에 합쳐 넣는 체인
from langchain_classic.chains import create_retrieval_chain  # 검색기와 답변 체인을 연결하는 RAG 체인
prompt = ChatPromptTemplate.from_template(
    """
    당신은 애널리스트입니다. 사용자의 질문과 해당 내용을 바탕으로 자세히 분석해서 알려주세요.

    반드시 아래 규칙을 지켜라.
    1. 답변은 무조건 한국어로 작성한다.
    2. 제공된 컨텍스트 범위 안에서만 답한다.
    3. 컨텍스트에 없는 내용은 추측하지 말고 "문서에서 확인되지 않습니다."라고 답한다.
    4. 감정 변화, 사건 흐름, 장면의 의미를 또렷하게 설명한다.
    5. 가능하면 마지막에 근거가 된 정보를 추출
    질문:
    {input}
    컨텍스트:
    {context}
    """
    )


In [49]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
load_dotenv()
import os


api_key = os.getenv("OLLAMA_API_KEY")


headers = {
   "Authorization": f"Bearer {api_key}"
}


llm = ChatOllama(
   base_url="https://ollama.com", # 원격 서버 주소
   model="gemma4:31b-cloud",
   client_kwargs={"headers": headers},
   temperature=0.2,
   reasoning=True
)




llm.invoke("hi")
llm.invoke("나 세차하러 갈껀데... 세차장까지 거리가 50미터야. 걸어야 가야할까? 운전해서 가야할까?")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'The user said "hi".\nThe user is initiating a conversation.\nRespond politely and offer assistance.\n\n*   "Hello! How can I help you today?"\n*   "Hi there! What\'s on your mind?"\n*   "Hello! Is there anything I can assist you with?"'}, response_metadata={'model': 'gemma4:31b-cloud', 'created_at': '2026-07-22T04:59:42.793306859Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1475617380, 'load_duration': None, 'prompt_eval_count': 17, 'prompt_eval_duration': None, 'eval_count': 77, 'eval_duration': None, 'logprobs': None, 'model_name': 'gemma4:31b-cloud', 'model_provider': 'ollama'}, id='lc_run--019f8831-9dfa-7903-97c9-9da4c0c59c79-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 77, 'total_tokens': 94})

In [50]:
llm.invoke("나 세차하러 갈껀데... 세차장까지 거리가 50미터야. 걸어야 가야할까? 운전해서 가야할까?")

AIMessage(content='세차하러 가시는 거라면... **당연히 운전해서 가셔야죠!** 🚗\n\n걸어가시면 세차장에 도착했을 때 **닦을 차가 없잖아요!** 😂\n\n50미터면 정말 가까운 거리니, 시동 걸고 천천히 이동해서 깨끗하게 세차하고 오세요! ✨', additional_kwargs={'reasoning_content': '*   Goal: Go to a car wash.\n    *   Distance: 50 meters.\n    *   Question: Walk or drive?\n\n    *   50 meters is very short (about 1/10th of a kilometer).\n    *   Walking time: Roughly 30-60 seconds.\n    *   Driving time: Starting the car, pulling out, driving 50m, parking/positioning. Probably takes longer or the same amount of time as walking.\n\n    *   *Scenario A: Walking.*\n        *   Pros: Fast, no need to start the engine, no traffic/parking stress.\n        *   Cons: You have to walk back *after* washing the car (but you\'ll be in the car then). Wait, the goal is to *wash* the car. You can\'t wash the car if you walk there.\n\n    *   *Scenario B: Driving.*\n        *   Pros: You have the car with you to actually wash it.\n        *   Cons: It\'s a very short distance.\n\n    *   Wai

In [51]:
document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, document_chain)


In [52]:
rt = rag_chain.invoke({'input' :'비트코인 전망'})

In [53]:
rt

{'input': '비트코인 전망',
 'context': [Document(id='5c5a0b00-2a0e-47dd-bfc1-6fb3937f20a7', metadata={'source': 'https://n.news.naver.com/mnews/article/421/0009069915'}, page_content='\n\n\n\n\n(유니티 제공)/뉴스1(서울=뉴스1) 김정현 기자 = 게임 엔진 유니티가 차세대 제작 플랫폼 \'유니티7\'(Unity 7)의 개발 로드맵을 공개했다.유니티는 21일 오전 서울 코엑스에서 글로벌 개발자 컨퍼런스 \'유나이트 서울 2026\'을 열고 차세대 제작 플랫폼 유니티7 관련 계획을 발표했다.유니티에 따르면 유니티7은 △더욱 빠른 제작 △개방형 협업 생태계 △획기적인 그래픽 △더욱 스마트해진 성장 및 수익화 △호환성 유지 5가지 핵심 축을 기반으로 구축됐다.유니티7은 개방형 협업 플랫폼을 통해 개발자, 아티스트, 프로듀서, 코딩 에이전트가 전체 개발 라이프사이클에 걸쳐 함께 작업할 수 있도록 지원하는 것을 특징으로 한다.개발자가 이미 사용하고 있는 AI 도구와도 긴밀하게 연동될 예정이다. 또 소규모 개발팀에게도 게임 수익화를 원활히 할 수 있도록 하는 기능도 탑재했다.마지막으로 유니티7은 기존 유니티6의 구조를 그대로 계승해 엔진 전환 과정에서 기존 작업물에 문제가 생기지 않도록 했다.이번 유니티7은 오는 12월 초기 베타 테스트에 들어간다. 정식 버전은 오는 2027년 1분기에 출시될 예정이다.매튜 브롬버그(Matthew Bromberg) 유니티 최고경영자(CEO)는 "이제 게임 개발의 미래는 가장 규모가 큰 개발팀이 아니라 새로운 기술로 고유한 콘텐츠를 만들고 플레이어를 확보할 수 있는 팀의 것이 될 것"이라며 "유니티7은 이같은 시대적 요구에 부응하는 플랫폼"이라고 강조했다.\n\t\t'),
  Document(id='8319a84b-66f3-4176-a685-5fd084a497e0', metadata={'source'

In [54]:
from kiwipiepy import Kiwi
kiwi = Kiwi()


In [55]:
kiwi.tokenize("아버지가방에들어가신다")

[Token(form='아버지', tag='NNG', start=0, len=3),
 Token(form='가', tag='JKS', start=3, len=1),
 Token(form='방', tag='NNG', start=4, len=1),
 Token(form='에', tag='JKB', start=5, len=1),
 Token(form='들어가', tag='VV', start=6, len=3),
 Token(form='시', tag='EP', start=9, len=1),
 Token(form='ᆫ다', tag='EF', start=9, len=2)]

In [56]:
retriever = vector_store.as_retriever(search_type='mmr', 
                                      search_kwargs={'fetch_k' : 10, 
                                      'lambda_mult' : 0.5}
                                      )
document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, document_chain)
rt = rag_chain.invoke({'input' :'삼성전자 전망'})


In [58]:
from langchain_community.retrievers import BM25Retriever


allowed_tags = {
        "NNG",  # 일반 명사
        "NNP",  # 고유 명사
        "VV",   # 동사
        "VA",   # 형용사
        "SL",   # 영문
        "SN",   # 숫자
    }


In [59]:
def kiwi_tokenize(text: str) -> list:
    """
    Kiwi를 이용해 한글 문장을 형태소 단위로 분리합니다.


    일반명사, 고유명사, 동사, 형용사, 영문, 숫자 등을
    검색 토큰으로 사용합니다.
    """
    allowed_tags = {
        "NNG",  # 일반 명사
        "NNP",  # 고유 명사
        "VV",   # 동사
        "VA",   # 형용사
        "SL",   # 영문
        "SN",   # 숫자
    }

    tokens = kiwi.tokenize(text)


    return [
        token.form.lower()
        for token in tokens
        if token.tag in allowed_tags
    ]


In [64]:
retriever_bm25 = BM25Retriever.from_documents(
    documents=data3,
    k=3,
    preprocess_func=kiwi_tokenize
)

In [65]:
retriever_bm25.invoke("삼성전자 주가 전망")

[Document(metadata={'source': 'https://n.news.naver.com/mnews/article/277/0005792503'}, page_content='\n외국인·기관 코스피 매수세…주가 상승세삼성전자·SK하닉도 각각 7%·6% 강세외국인과 기관의 매수세로 코스피지수가 4%대 상승 중이다. 지수 급등으로 한때 매수 사이드카(프로그램 매수호가 일시 효력정지)가 발동하기도 했다.\n\n\n\n연합뉴스21일 오후 1시50분 기준 코스피지수는 전 거래일 대비 305.29포인트(4.69%) 오른 6821.56에 거래되고 있다. 투자자별로는 외국인과 기관이 각각 5021억7700만원, 1조8888억100만원 순매수 중이며, 개인은 2조3463억3500만원 순매도 중이다.이날 오후 12시41분29초부터 코스피200선물지수가 전일 종가보다 53.44포인트(5.17%) 오른 1086.58을 기록하면서 5분간 매수 사이드카가 발동하기도 했다.업종별로는 종이·목재(-0.71%), 비금속(-0.46%), 금속(-0.22%)을 제외한 전 업종이 강세다. 전기·전자(6.18%), 제조(5.12%), 유통(5.38%), 전기·가스(3.23%) 등은 3% 이상 상승률을 보인다.시가총액 1위인 삼성전자(7.1%)와 2위인 SK하이닉스(6.0%)는 높은 상승률을 보이고 있고 삼성물산(8.1%), SK스퀘어(6.8%), 삼성바이오로직스(4.7%), 삼성전기(3.2%), 삼성생명(3.1%) 등도 강세다.같은 시각 코스닥지수는 전 거래일 대비 6.01포인트(0.80%) 오른 755.65에 거래 중이다. 외국인과 기관은 각각 1579억6200만원, 174억8600만원 순매도 중이며 개인은 1704억5400만원 순매수 중이다.업종별로는 제약(-3.08%), 출판·매체복제(-1.99%), 일반서비스(-0.76%), 섬유·의류(-0.66%)를 제외한 모든 업종이 상승세다. 건설(2.41%), 유통(2.24%), 기계·장비(2.33%), 전기·전자(2.06%) 등이 강세를 보

In [66]:
document_chain = create_stuff_documents_chain(llm, prompt)
rag_bm25_chain = create_retrieval_chain(retriever_bm25, document_chain)


In [67]:
rt = rag_bm25_chain.invoke({'input' :'삼성전자 전망'})

In [68]:
print(rt['answer'])

제시된 컨텍스트를 바탕으로 분석한 삼성전자의 전망은 다음과 같습니다.

### [삼성전자 분석 보고서]

**1. 전략적 방향: AI 시대의 핵심 성장동력 '로봇 사업' 본격 육성**
삼성전자는 AI 시대를 맞아 로봇 사업을 차세대 핵심 성장동력으로 설정하고, 이를 위한 통합 추진체계를 구축하며 공격적인 행보를 보이고 있습니다.

*   **조직 및 체계의 변화:** 대표이사(노태문 사장) 직속의 **‘RX(Robotics eXperience)사업추진실’**을 신설했습니다. 이는 단순한 연구 단계를 넘어 중장기 전략 수립, 기술 개발, 디자인, 상품기획까지 아우르는 '전담 컨트롤타워'를 통해 의사결정 속도를 높이고 실행력을 극대화하려는 의도로 풀이됩니다.
*   **인적·물적 인프라 확충:** 
    *   **인재 영입:** 현대차그룹 출신의 이동건 부사장을 비롯해 지능형 자율 시스템 및 인간형 로봇 손 분야의 세계적 석학(김현진, 김의겸 교수)을 영입하여 기술적 전문성을 강화했습니다.
    *   **거점 확대:** 서울R&D캠퍼스(협업 환경), 구미사업장(데이터 팩토리), 그리고 미국·중국·일본의 글로벌 연구거점을 통해 현지 기술 확보와 인재 영입을 추진하고 있습니다.
*   **상용화 로드맵:** **‘제조 현장 선(先) 검증 $\rightarrow$ B2B·B2C 단계적 확장’**이라는 구체적인 경로를 설정했습니다. 특히 2030년까지 전 세계 생산 공장을 ‘AI 자율공장’으로 전환하여 오퍼레이팅·물류·조립봇의 정밀도와 범용성을 높이는 전략을 가동 중입니다.
*   **핵심 경쟁력:** 삼성전자뿐만 아니라 삼성디스플레이, 삼성전기, 삼성SDI, 삼성중공업 등 관계사들이 보유한 다양한 제조 현장의 데이터를 학습할 수 있다는 점이 범용 로봇 진화의 강력한 무기로 분석됩니다.

**2. 시장 반응 및 주가 동향**
삼성전자의 이러한 전략적 움직임과 시장 상황이 맞물려 긍정적인 주가 흐름을 보이고 있습니다.

*   **주가 상승:** 외국인과 기관의 강

In [112]:
client_id="sYf288UqKIwCM4woD3GW"
client_secret="qakMH4VRbC"


In [84]:
template = """
사용자가 질문하는 내용을 이해하고, 그에 따라 중요한 키워드 추출할 것
키워드 구분은 콤마로 할 것

원본 질문 : {question}
"""
prompt = ChatPromptTemplate.from_template(template)

chain = prompt | llm | StrOutputParser() | (lambda x : x.split(","))

In [85]:
rt = chain.invoke({'question' : 'kimi-3가 전세계 AI시장에 미치는 영향 분석해줘'})

In [86]:
rt

['kimi-3', ' 전세계', ' AI시장', ' 영향', ' 분석']

In [87]:
tmp = get_news(rt)

In [89]:
Document(page_content=tmp[0]['description'], metadata={'title' : tmp[0]['title'],
                                                       'link' :tmp[0]['originallink'],
                                                       'date' : tmp[0]['pubDate']})


Document(metadata={'title': '제품 수출은 옛말…중국 AI 기업, 이젠 ‘토큰’ 수출한다 [여기는 중국...', 'link': 'https://nownews.seoul.co.kr/news/newsView.php?id=20260722601020&wlog_tag3=naver', 'date': 'Wed, 22 Jul 2026 15:52:00 +0900'}, page_content='딥시크와 알리바바 큐원(Qwen), 키미(<b>Kimi</b>), 즈푸AI, 미니맥스(MiniMax) 등 중국의 주요 AI 기업들은 자사 모델을 잇달아 개방하고 있다. 중국산 AI 반도체의 활용 범위도 넓어지고 있다. WAIC 참가 기업들에 따르면 화웨이 등... ')

In [92]:
def get_news(query: list , loop: int = 5) -> list[Document]:
    url = "https://openapi.naver.com/v1/search/news.json"
    headers = {
        "X-Naver-Client-Id": client_id,
        "X-Naver-Client-Secret": client_secret
    }
    total = []
    for q in query:
        for x in range(1,loop+1):
            params = {'query' : q, 'display' : 100, 'start': x, "sort" : "date"}
            total.extend(requests.get(url, headers=headers, params=params).json()['items'])
    
    return  [Document(page_content=x['description'], metadata={'title' : x['title'], 
                                                       'link' :x['originallink'], 
                                                       'date' : x['pubDate']}) for x in total]


In [104]:
tmp = get_news(rt)

In [108]:
retriever_bm25 = BM25Retriever.from_documents(
    documents=tmp,
    k=20,
    preprocess_func=kiwi_tokenize
)
prompt = ChatPromptTemplate.from_template(
    """
    당신은 애널리스트입니다. 사용자의 질문과 해당 내용을 바탕으로 자세히 분석해서 알려주세요.

    반드시 아래 규칙을 지켜라.
    1. 답변은 무조건 한국어로 작성한다.
    2. 제공된 컨텍스트 범위 안에서만 답한다.
    3. 컨텍스트에 없는 내용은 추측하지 말고 "문서에서 확인되지 않습니다."라고 답한다.
    4. 감정 변화, 사건 흐름, 장면의 의미를 또렷하게 설명한다.
    5. 가능하면 마지막에 근거가 된 정보를 추출
    질문:
    {input}
    컨텍스트:
    {context}
    """
    )


In [109]:
document_chain = create_stuff_documents_chain(llm, prompt)
rag_bm25_chain = create_retrieval_chain(retriever_bm25, document_chain)

In [110]:
rt = rag_bm25_chain.invoke({'input' : 'kimi-3가 전세계 AI 시장에 미치는 영향 분석해줘'})


In [111]:
rt

{'input': 'kimi-3가 전세계 AI 시장에 미치는 영향 분석해줘',
 'context': [Document(metadata={'title': '[크립토 브리핑] 반도체 쇼크 진정에 비트코인 2% 반등…ETF 자금도 5일째...', 'link': 'https://www.techm.kr/news/articleView.html?idxno=153553', 'date': 'Wed, 22 Jul 2026 10:00:00 +0900'}, page_content='리플(XRP)도 약 <b>3</b>% 상승했고 솔라나 역시 2% 안팎의 오름세를 기록하며 위험자산 선호 심리 회복에 동참했다. 시장 반등에는 아시아 증시 안정이 영향을 미친 것으로 분석된다. 지난주 키미(<b>Kimi</b>) AI 충격으로 급락했던... '),
  Document(metadata={'title': '[크립토 브리핑] 반도체 쇼크 진정에 비트코인 2% 반등…ETF 자금도 5일째...', 'link': 'https://www.techm.kr/news/articleView.html?idxno=153553', 'date': 'Wed, 22 Jul 2026 10:00:00 +0900'}, page_content='리플(XRP)도 약 <b>3</b>% 상승했고 솔라나 역시 2% 안팎의 오름세를 기록하며 위험자산 선호 심리 회복에 동참했다. 시장 반등에는 아시아 증시 안정이 영향을 미친 것으로 분석된다. 지난주 키미(<b>Kimi</b>) AI 충격으로 급락했던... '),
  Document(metadata={'title': '[크립토 브리핑] 반도체 쇼크 진정에 비트코인 2% 반등…ETF 자금도 5일째...', 'link': 'https://www.techm.kr/news/articleView.html?idxno=153553', 'date': 'Wed, 22 Jul 2026 10:00:00 +0900'}, page_content='리플(XRP)도 약 <b>3</b>% 상승했고 솔라나 역시 2% 